# Notebook 06: Parallel Execution, Subgraphs, and Map-Reduce Patterns

## Learning Objectives

1. Implement parallel execution with the Send API
2. Create subgraphs for modularity
3. Build multi-agent architectures
4. Use map-reduce patterns
5. Implement deferred nodes (LangGraph 1.1.9 feature)

## Prerequisites

- Completed Notebooks 01-05
- Understanding of agents, tools, and state management

## 1. Introduction to Parallel Execution

### Why Parallel Execution?

Process multiple items simultaneously:
- 🚀 **Faster execution** - Don't wait for sequential processing
- 📊 **Batch operations** - Handle multiple requests at once
- 🔄 **Map-reduce workflows** - Process and aggregate results

### The Send API

```python
from langgraph.types import Send

# Instead of returning a node name, return Send objects:
return [Send("process_item", {"item": item}) for item in items]
```

This creates **multiple parallel executions** of the same node!

## 🔷 Send API — Why Two States?

```mermaid
flowchart TD

    OS1["🌐 OverallState  ·  global shared state
    ─────────────────────────────────
    items: A, B, C
    results: []"]

    DF["dispatch_function
    reads OverallState
    returns  Send · Send · Send"]

    OS1 --> DF

    WS1["🔒 WorkerState 1
    ──────────────────
    item: A
    extra_ctx: ...
    private only"]

    WS2["🔒 WorkerState 2
    ──────────────────
    item: B
    extra_ctx: ...
    private only"]

    WS3["🔒 WorkerState 3
    ──────────────────
    item: C
    extra_ctx: ...
    private only"]

    DF -->|"Send('worker', {item: A})"| WS1
    DF -->|"Send('worker', {item: B})"| WS2
    DF -->|"Send('worker', {item: C})"| WS3

    W1["⚙️ worker node
    processes A in isolation"]

    W2["⚙️ worker node
    processes B in isolation"]

    W3["⚙️ worker node
    processes C in isolation"]

    WS1 --> W1
    WS2 --> W2
    WS3 --> W3

    RD["🔀 Reducer  ·  Annotated[list, operator.add]
    safely merges all parallel results"]

    W1 -->|"return {'results': ['A_out']}"| RD
    W2 -->|"return {'results': ['B_out']}"| RD
    W3 -->|"return {'results': ['C_out']}"| RD

    OS2["🌐 OverallState  ·  updated
    ─────────────────────────────────
    items: A, B, C
    results: A_out, B_out, C_out"]

    RD --> OS2

    style OS1  fill:#EEEDFE,stroke:#534AB7,color:#26215C
    style OS2  fill:#EEEDFE,stroke:#534AB7,color:#26215C
    style DF   fill:#F1EFE8,stroke:#5F5E5A,color:#2C2C2A
    style WS1  fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style WS2  fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style WS3  fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style W1   fill:#FAECE7,stroke:#993C1D,color:#4A1B0C
    style W2   fill:#FAECE7,stroke:#993C1D,color:#4A1B0C
    style W3   fill:#FAECE7,stroke:#993C1D,color:#4A1B0C
    style RD   fill:#FAEEDA,stroke:#854F0B,color:#412402
```

### Key takeaways

| | OverallState | WorkerState |
|---|---|---|
| **Scope** | Entire graph | One worker only |
| **Who creates it** | You (initial state) | `Send()` injects it |
| **Who sees it** | Every node | That worker alone |
| **How results return** | Via `Annotated` reducer | `return {"field": [value]}` |
| **Purpose** | Shared memory | Private task input |

> **Rule:** Workers never read `OverallState` directly. They only get what `Send()` gives them.
> Results flow back only through the reducer — never by direct write.

In [1]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import operator

load_dotenv()
print("✅ Ready for parallel execution!")

✅ Ready for parallel execution!


## 2. Example 1: Parallel Data Processor

Process multiple items in parallel using the Send API.

In [16]:
class BatchState(TypedDict):
    items: list  # Items to process
    results: Annotated[list, operator.add]  # Results accumulate

class ItemState(TypedDict):
    item: str  # Single item

def distribute_work(state: BatchState):
    """Distribute items for parallel processing."""
    print(f"📦 Distributing {len(state['items'])} items for processing")
    return [Send("process_item", {"item": item}) for item in state["items"]]

def process_item(state: ItemState) -> dict:
    """Process a single item."""
    item = state["item"]
    print(f"  ⚙️  Processing: {item}")
    result = f"Processed: {item.upper()}"
    return {"results": [result]}

# Build parallel processing graph
parallel_graph = StateGraph(BatchState)
parallel_graph.add_node("process_item", process_item)

# distribute_work goes here as the conditional edge function, not a node
parallel_graph.add_conditional_edges(START, distribute_work, ["process_item"])
parallel_graph.add_edge("process_item", END)

parallel_app = parallel_graph.compile()

# Test parallel processing
result = parallel_app.invoke({
    "items": ["apple", "banana", "cherry", "date"],
    "results": []
})

print("\n📊 Results:")
for r in result["results"]:
    print(f"  • {r}")

📦 Distributing 4 items for processing
  ⚙️  Processing: apple
  ⚙️  Processing: banana
  ⚙️  Processing: cherry
  ⚙️  Processing: date

📊 Results:
  • Processed: APPLE
  • Processed: BANANA
  • Processed: CHERRY
  • Processed: DATE


In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# ── Two-level state design ────────────────────────────────
# OverallState: the global shared state of the whole graph
# WorkerState:  the private state injected per Send() call

class OverallState(TypedDict):
    documents:  list[str]           # input: list of docs to review
    reviews:   Annotated[list[str], operator.add]  # reducer: collects all results
    summary:   str

class WorkerState(TypedDict):
    document:  str   # ← each worker gets ONE document (not the full list)
    doc_index: int   # ← worker also knows which doc it is

# ── Nodes ─────────────────────────────────────────────────
def load_documents(state: OverallState):
    """Entry node: loads 4 documents into global state."""
    docs = [
        "Q4 Revenue Report: revenue up 18% YoY",
        "Risk Assessment: supply chain risks identified",
        "Product Roadmap: 3 major features planned for H1",
        "HR Update: headcount grew by 15 in Q4",
    ]
    print(f"📂 Loaded {len(docs)} documents.")
    return {"documents": docs}

def review_document(state: WorkerState):
    """
    Worker node — receives ONE document via Send().
    Note: 'state' here is WorkerState, NOT OverallState.
    Each parallel invocation of this node has a different 'document'.
    """
    doc   = state["document"]
    idx   = state["doc_index"]
    review = f"[Doc {idx+1}] ✅ Reviewed: {doc[:40]}..."
    print(f"  🔎 Worker {idx+1} reviewing doc...")
    return {"reviews": [review]}  # → appended to OverallState.reviews via reducer

def compile_summary(state: OverallState):
    """Fan-in node: receives all reviews merged by the reducer."""
    print(f"\n📝 All {len(state['reviews'])} reviews collected. Compiling summary...")
    summary = "SUMMARY: " + " | ".join(state["reviews"])
    return {"summary": summary}

# ── The KEY function: returns a list of Send() objects ────
def dispatch_reviewers(state: OverallState) -> list[Send]:
    """
    Called as a conditional edge function.
    Returns N Send objects → spawns N parallel review_document workers,
    each with its OWN custom state dict (different document + index).
    """
    return [
        Send("review_document", {"document": doc, "doc_index": i})
        for i, doc in enumerate(state["documents"])
    ]
    # Result: 4 parallel invocations of review_document, each with different doc

# ── Build Graph ───────────────────────────────────────────
builder = StateGraph(OverallState)
builder.add_node("load_documents",   load_documents)
builder.add_node("review_document",   review_document)
builder.add_node("compile_summary",   compile_summary)
builder.add_edge(START,               "load_documents")

# ← dispatch_reviewers returns [Send(...), Send(...), ...]
builder.add_conditional_edges("load_documents", dispatch_reviewers, ["review_document"])
builder.add_edge("review_document",   "compile_summary")
builder.add_edge("compile_summary",   END)
graph = builder.compile()

result = graph.invoke({"documents": [], "reviews": [], "summary": ""})
print(f"\n✅ Final summary:\n{result['summary']}")

### SEND API MORE EXAMPLES

In [1]:
# ============================================================
# SEND API — EXAMPLE 1: DIFFERENT NODE PER ITEM
# Most people think Send only fans out to ONE node type.
# But each Send() call can target a DIFFERENT node.
# Here: each ticket routes to a specialist node based on its type.
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class OverallState(TypedDict):
    tickets:  list[dict]
    resolved: Annotated[list[str], operator.add]

class TicketState(TypedDict):
    ticket_id: str
    message:  str
    priority: str

# ── Three different specialist nodes ─────────────────────
def handle_billing(state: TicketState):
    print(f"  💳 BILLING  agent → ticket {state['ticket_id']}: '{state['message'][:30]}'")
    return {"resolved": [f"[BILLING]  {state['ticket_id']} resolved"]}

def handle_tech(state: TicketState):
    print(f"  🔧 TECH     agent → ticket {state['ticket_id']}: '{state['message'][:30]}'")
    return {"resolved": [f"[TECH]     {state['ticket_id']} resolved"]}

def handle_escalation(state: TicketState):
    print(f"  🚨 ESCALATE agent → ticket {state['ticket_id']}: '{state['message'][:30]}' [PRIORITY: {state['priority']}]")
    return {"resolved": [f"[ESCALATE] {state['ticket_id']} escalated to manager"]}

def load_tickets(state: OverallState):
    tickets = [
        {"ticket_id": "T001", "type": "billing",    "priority": "low",      "message": "I was charged twice this month"},
        {"ticket_id": "T002", "type": "tech",       "priority": "medium",   "message": "App crashes on startup"},
        {"ticket_id": "T003", "type": "billing",    "priority": "critical", "message": "Fraudulent charge on account"},
        {"ticket_id": "T004", "type": "tech",       "priority": "low",      "message": "Dark mode not working"},
        {"ticket_id": "T005", "type": "tech",       "priority": "critical", "message": "Data loss after update"},
    ]
    print(f"📥 Loaded {len(tickets)} support tickets.")
    return {"tickets": tickets}

# ── KEY: Each item dispatches to a DIFFERENT node ─────────
def dispatch_tickets(state: OverallState) -> list[Send]:
    sends = []
    for t in state["tickets"]:
        worker_state = {"ticket_id": t["ticket_id"],
                        "message":   t["message"],
                        "priority":  t["priority"]}

        # CRITICAL tickets → escalation node (regardless of type)
        # billing tickets  → billing node
        # tech tickets     → tech node
        if t["priority"] == "critical":
            sends.append(Send("handle_escalation", worker_state))  # ← different node!
        elif t["type"] == "billing":
            sends.append(Send("handle_billing",    worker_state))  # ← different node!
        else:
            sends.append(Send("handle_tech",       worker_state))  # ← different node!
    print(f"\n🚦 Dispatching {len(sends)} tickets to different specialist nodes in parallel...")
    return sends

def report(state: OverallState):
    print(f"\n📊 All {len(state['resolved'])} tickets resolved:")
    for r in sorted(state["resolved"]):
        print(f"   {r}")
    return {}

builder = StateGraph(OverallState)
builder.add_node("load_tickets",      load_tickets)
builder.add_node("handle_billing",    handle_billing)
builder.add_node("handle_tech",       handle_tech)
builder.add_node("handle_escalation", handle_escalation)
builder.add_node("report",             report)
builder.add_edge(START,                "load_tickets")
builder.add_conditional_edges(
    "load_tickets", dispatch_tickets,
    ["handle_billing", "handle_tech", "handle_escalation"]  # ← declare all possible targets
)
# All 3 specialist nodes converge at report
builder.add_edge("handle_billing",    "report")
builder.add_edge("handle_tech",       "report")
builder.add_edge("handle_escalation", "report")
builder.add_edge("report",             END)

graph = builder.compile()
graph.invoke({"tickets": [], "resolved": []})

📥 Loaded 5 support tickets.

🚦 Dispatching 5 tickets to different specialist nodes in parallel...
  💳 BILLING  agent → ticket T001: 'I was charged twice this month'
  🔧 TECH     agent → ticket T002: 'App crashes on startup'
  🚨 ESCALATE agent → ticket T003: 'Fraudulent charge on account' [PRIORITY: critical]
  🔧 TECH     agent → ticket T004: 'Dark mode not working'
  🚨 ESCALATE agent → ticket T005: 'Data loss after update' [PRIORITY: critical]

📊 All 5 tickets resolved:
   [BILLING]  T001 resolved
   [ESCALATE] T003 escalated to manager
   [ESCALATE] T005 escalated to manager
   [TECH]     T002 resolved
   [TECH]     T004 resolved


{'tickets': [{'ticket_id': 'T001',
   'type': 'billing',
   'priority': 'low',
   'message': 'I was charged twice this month'},
  {'ticket_id': 'T002',
   'type': 'tech',
   'priority': 'medium',
   'message': 'App crashes on startup'},
  {'ticket_id': 'T003',
   'type': 'billing',
   'priority': 'critical',
   'message': 'Fraudulent charge on account'},
  {'ticket_id': 'T004',
   'type': 'tech',
   'priority': 'low',
   'message': 'Dark mode not working'},
  {'ticket_id': 'T005',
   'type': 'tech',
   'priority': 'critical',
   'message': 'Data loss after update'}],
 'resolved': ['[BILLING]  T001 resolved',
  '[TECH]     T002 resolved',
  '[ESCALATE] T003 escalated to manager',
  '[TECH]     T004 resolved',
  '[ESCALATE] T005 escalated to manager']}

In [2]:
# ============================================================
# SEND API — EXAMPLE 2: VARIABLE FAN-OUT COUNT
# The number of Send() calls is NOT fixed at build time.
# It is computed DYNAMICALLY from the current state at runtime.
# Example: Exam grader — spawns one grader per student (0 to N).
# Also shows: passing EXTRA metadata alongside the item.
# ============================================================

import operator, random
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class ExamState(TypedDict):
    exam_name:    str
    total_marks:  int
    submissions:  list[dict]           # dynamically filled
    grades:       Annotated[list[dict], operator.add]

class GraderState(TypedDict):
    student_id:   str
    answers:      list[int]           # raw scores per question
    exam_name:    str                 # ← extra metadata passed from OverallState
    total_marks:  int                 # ← extra metadata passed from OverallState

# ── Phase 1: Simulate collecting submissions ──────────────
def collect_submissions(state: ExamState):
    """
    Simulates receiving N submissions. N is random (3-7).
    The fan-out count will only be known HERE, at runtime.
    """
    random.seed(42)
    n_students   = random.randint(4, 7)
    submissions  = [
        {
            "student_id": f"STU-{i+1:03d}",
            "answers":    [random.randint(0, 10) for _ in range(5)]  # 5 questions
        }
        for i in range(n_students)
    ]
    print(f"📋 Collected {n_students} submissions for '{state['exam_name']}'")
    return {"submissions": submissions}

# ── Dispatch: VARIABLE number of graders ──────────────────
def dispatch_graders(state: ExamState) -> list[Send]:
    """
    Returns len(submissions) Send objects — one per student.
    Each worker also receives exam_name and total_marks as context.
    The exact number is ONLY known here at runtime!
    """
    n = len(state["submissions"])
    print(f"⚡ Spawning {n} grader workers dynamically...")
    return [
        Send("grade_submission", {
            "student_id":  sub["student_id"],
            "answers":     sub["answers"],
            # ↓ EXTRA METADATA passed alongside the item
            "exam_name":   state["exam_name"],
            "total_marks": state["total_marks"],
        })
        for sub in state["submissions"]
    ]

# ── Worker: grades one student's answers ─────────────────
def grade_submission(state: GraderState):
    raw       = sum(state["answers"])
    pct       = round(raw / state["total_marks"] * 100, 1)
    grade     = "A" if pct>=80 else "B" if pct>=65 else "C" if pct>=50 else "F"
    result    = {"student":state["student_id"], "score":raw, "pct":pct, "grade":grade}
    print(f"  📝 {state['student_id']}: {raw}/{state['total_marks']} = {pct}% → Grade {grade}")
    return {"grades": [result]}

# ── Reduce: compute class statistics ──────────────────────
def compute_stats(state: ExamState):
    grades  = state["grades"]
    avg     = round(sum(g["pct"] for g in grades) / len(grades), 1)
    top     = max(grades, key=lambda x: x["pct"])
    dist    = {}
    for g in grades:
        dist[g["grade"]] = dist.get(g["grade"], 0) + 1
    print(f"\n📊 {state['exam_name']} Results ({len(grades)} students):")
    print(f"   Class average : {avg}%")
    print(f"   Top student   : {top['student']} ({top['pct']}%)")
    print(f"   Grade dist    : {dist}")
    return {}

builder = StateGraph(ExamState)
builder.add_node("collect_submissions", collect_submissions)
builder.add_node("grade_submission",    grade_submission)
builder.add_node("compute_stats",       compute_stats)
builder.add_edge(START,                  "collect_submissions")
builder.add_conditional_edges("collect_submissions", dispatch_graders, ["grade_submission"])
builder.add_edge("grade_submission", "compute_stats")
builder.add_edge("compute_stats",    END)
graph = builder.compile()

graph.invoke({
    "exam_name":   "Advanced LLMs & RAG Midterm",
    "total_marks": 50,
    "submissions": [],
    "grades":      [],
})

📋 Collected 4 submissions for 'Advanced LLMs & RAG Midterm'
⚡ Spawning 4 grader workers dynamically...
  📝 STU-001: 12/50 = 24.0% → Grade F
  📝 STU-002: 29/50 = 58.0% → Grade C
  📝 STU-003: 10/50 = 20.0% → Grade F
  📝 STU-004: 28/50 = 56.0% → Grade C

📊 Advanced LLMs & RAG Midterm Results (4 students):
   Class average : 39.5%
   Top student   : STU-002 (58.0%)
   Grade dist    : {'F': 2, 'C': 2}


{'exam_name': 'Advanced LLMs & RAG Midterm',
 'total_marks': 50,
 'submissions': [{'student_id': 'STU-001', 'answers': [0, 4, 3, 3, 2]},
  {'student_id': 'STU-002', 'answers': [1, 10, 8, 1, 9]},
  {'student_id': 'STU-003', 'answers': [6, 0, 0, 1, 3]},
  {'student_id': 'STU-004', 'answers': [3, 8, 9, 0, 8]}],
 'grades': [{'student': 'STU-001', 'score': 12, 'pct': 24.0, 'grade': 'F'},
  {'student': 'STU-002', 'score': 29, 'pct': 58.0, 'grade': 'C'},
  {'student': 'STU-003', 'score': 10, 'pct': 20.0, 'grade': 'F'},
  {'student': 'STU-004', 'score': 28, 'pct': 56.0, 'grade': 'C'}]}

In [5]:
# ============================================================
# SEND API — EXAMPLE 3 FIXED: CHAINED SEND (TWO-LEVEL FAN-OUT)
# 
# WHAT WAS WRONG:
# audit_department was used as BOTH a node (must return dict)
# AND a routing function (returns [Send(...)]). That's invalid.
#
# THE FIX:
# Introduce an intermediate fan-in node (collect_depts) that
# waits for ALL level 1 workers, then a separate dispatch
# function for level 2. Clean separation of concerns.
#
# Correct flow:
# start_audit → [dispatch_departments] → audit_department ×3 (parallel)
#                                               ↓ (fan-in)
#                                         collect_depts
#                                               ↓ [dispatch_employees]
#                                         audit_employee ×9 (parallel)
#                                               ↓ (fan-in)
#                                         compile_report → END
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# ── States ────────────────────────────────────────────────
class AuditState(TypedDict):
    company:   str
    dept_data: Annotated[list[dict], operator.add]  # level 1 workers write here
    audit_log: Annotated[list[str],  operator.add]  # all workers append here

class DeptState(TypedDict):  # private state for level 1 workers
    company:   str
    dept:      str
    employees: list[str]

class EmpState(TypedDict):   # private state for level 2 workers
    company:   str
    dept:      str
    employee:  str

# ── Level 0: Entry node ───────────────────────────────────
def start_audit(state: AuditState):
    print(f"🏢 Starting HR audit for: {state['company']}")
    return {"audit_log": [f"Audit started for {state['company']}"]}

# ── Level 0 → Level 1 dispatch ────────────────────────────
def dispatch_departments(state: AuditState) -> list[Send]:
    """FIRST fan-out: one Send per department."""
    departments = {
        "Engineering": ["Alice", "Bob", "Charlie"],
        "Marketing":   ["Diana", "Eve"],
        "Finance":     ["Frank", "Grace", "Henry", "Ivy"],
    }
    print(f"\n  📂 Fan-out LEVEL 1: dispatching {len(departments)} dept auditors in parallel...")
    return [
        Send("audit_department", {
            "company":   state["company"],
            "dept":      dept,
            "employees": emps,
        })
        for dept, emps in departments.items()
    ]

# ── Level 1 worker: RETURNS A DICT (not Send objects!) ────
def audit_department(state: DeptState):
    """
    FIX: This node now correctly returns a dict.
    It stores employee data into 'dept_data' via the reducer
    so the level 2 dispatcher can read them later.
    """
    print(f"  🗂  [{state['dept']}] auditing dept — {len(state['employees'])} employees found")
    return {
        "dept_data": [{            # ← written to global AuditState via operator.add
            "company":   state["company"],
            "dept":      state["dept"],
            "employees": state["employees"],
        }],
        "audit_log": [f"Dept audited: {state['dept']} ({len(state['employees'])} employees)"],
    }

# ── Intermediate fan-in: waits for ALL level 1 workers ────
def collect_depts(state: AuditState):
    """
    FIX: This new intermediate node runs AFTER all audit_department
    workers complete and their results are merged into AuditState.
    It's a passthrough — just confirms collection before level 2 dispatch.
    """
    total = sum(len(d["employees"]) for d in state["dept_data"])
    print(f"\n  ✅ All {len(state['dept_data'])} depts collected. Total employees: {total}")
    return {}

# ── Level 1 → Level 2 dispatch ────────────────────────────
def dispatch_employees(state: AuditState) -> list[Send]:
    """
    SECOND fan-out: reads dept_data that was collected from all
    level 1 workers and spawns one Send per employee.
    """
    sends = [
        Send("audit_employee", {
            "company":  d["company"],
            "dept":     d["dept"],
            "employee": emp,
        })
        for d in state["dept_data"]
        for emp in d["employees"]
    ]
    print(f"  📂 Fan-out LEVEL 2: dispatching {len(sends)} employee auditors in parallel...")
    return sends

# ── Level 2 worker: leaf node, returns dict ───────────────
def audit_employee(state: EmpState):
    entry = f"✓ {state['company']} | {state['dept']:12s} | {state['employee']}"
    print(f"    👤 {entry}")
    return {"audit_log": [entry]}

# ── Final fan-in: compile report ──────────────────────────
def compile_report(state: AuditState):
    emp_logs = [l for l in state["audit_log"] if l.startswith("✓")]
    print(f"\n📋 AUDIT COMPLETE — {len(emp_logs)} employees audited:")
    for log in sorted(emp_logs):
        print(f"   {log}")
    return {}

# ── Build Graph ───────────────────────────────────────────
builder = StateGraph(AuditState)
builder.add_node("start_audit",       start_audit)
builder.add_node("audit_department",  audit_department)   # level 1: returns dict ✅
builder.add_node("collect_depts",     collect_depts)      # intermediate fan-in node ✅
builder.add_node("audit_employee",    audit_employee)     # level 2: returns dict ✅
builder.add_node("compile_report",    compile_report)

builder.add_edge(START,                "start_audit")
# Level 1 fan-out
builder.add_conditional_edges("start_audit",   dispatch_departments, ["audit_department"])
# Fan-in: all level 1 workers → collect_depts
builder.add_edge("audit_department",           "collect_depts")
# Level 2 fan-out
builder.add_conditional_edges("collect_depts", dispatch_employees,   ["audit_employee"])
# Fan-in: all level 2 workers → compile_report
builder.add_edge("audit_employee",             "compile_report")
builder.add_edge("compile_report",             END)

graph = builder.compile()
graph.invoke({"company": "KrishAI Technologies", "dept_data": [], "audit_log": []})

🏢 Starting HR audit for: KrishAI Technologies

  📂 Fan-out LEVEL 1: dispatching 3 dept auditors in parallel...
  🗂  [Engineering] auditing dept — 3 employees found
  🗂  [Marketing] auditing dept — 2 employees found
  🗂  [Finance] auditing dept — 4 employees found

  ✅ All 3 depts collected. Total employees: 9
  📂 Fan-out LEVEL 2: dispatching 9 employee auditors in parallel...
    👤 ✓ KrishAI Technologies | Engineering  | Alice
    👤 ✓ KrishAI Technologies | Engineering  | Bob
    👤 ✓ KrishAI Technologies | Engineering  | Charlie
    👤 ✓ KrishAI Technologies | Marketing    | Diana
    👤 ✓ KrishAI Technologies | Marketing    | Eve
    👤 ✓ KrishAI Technologies | Finance      | Frank
    👤 ✓ KrishAI Technologies | Finance      | Grace
    👤 ✓ KrishAI Technologies | Finance      | Henry
    👤 ✓ KrishAI Technologies | Finance      | Ivy

📋 AUDIT COMPLETE — 9 employees audited:
   ✓ KrishAI Technologies | Engineering  | Alice
   ✓ KrishAI Technologies | Engineering  | Bob
   ✓ KrishAI Technol

{'company': 'KrishAI Technologies',
 'dept_data': [{'company': 'KrishAI Technologies',
   'dept': 'Engineering',
   'employees': ['Alice', 'Bob', 'Charlie']},
  {'company': 'KrishAI Technologies',
   'dept': 'Marketing',
   'employees': ['Diana', 'Eve']},
  {'company': 'KrishAI Technologies',
   'dept': 'Finance',
   'employees': ['Frank', 'Grace', 'Henry', 'Ivy']}],
 'audit_log': ['Audit started for KrishAI Technologies',
  'Dept audited: Engineering (3 employees)',
  'Dept audited: Marketing (2 employees)',
  'Dept audited: Finance (4 employees)',
  '✓ KrishAI Technologies | Engineering  | Alice',
  '✓ KrishAI Technologies | Engineering  | Bob',
  '✓ KrishAI Technologies | Engineering  | Charlie',
  '✓ KrishAI Technologies | Marketing    | Diana',
  '✓ KrishAI Technologies | Marketing    | Eve',
  '✓ KrishAI Technologies | Finance      | Frank',
  '✓ KrishAI Technologies | Finance      | Grace',
  '✓ KrishAI Technologies | Finance      | Henry',
  '✓ KrishAI Technologies | Finance   

In [4]:
# ============================================================
# SEND API — EXAMPLE 4: WORKERS ROUTE TO DIFFERENT DOWNSTREAM NODES
# Combines Send API (for fan-out) with Command(goto=...) inside
# the worker node to dynamically pick the NEXT node per item.
# Pipeline: detect content type → parallel workers → each worker
# routes to its own post-processing node based on result quality.
# Example: RAG pipeline — retrieve N chunks, each gets scored,
# high-score chunks go to "use_chunk", low-score to "discard_chunk".
# ============================================================

import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, Command

class RAGState(TypedDict):
    query:    str
    chunks:   list[dict]
    used:     Annotated[list[str], operator.add]   # kept chunks
    discarded:Annotated[list[str], operator.add]   # dropped chunks
    answer:   str

class ChunkState(TypedDict):
    query:    str
    chunk_id: str
    text:     str
    score:    float   # relevance score pre-computed

# ── Entry: simulate retrieval ─────────────────────────────
def retrieve_chunks(state: RAGState):
    chunks = [
        {"chunk_id":"C1","text":"LangGraph supports stateful multi-agent workflows","score":0.91},
        {"chunk_id":"C2","text":"The weather in Paris is 22 degrees today",       "score":0.12},
        {"chunk_id":"C3","text":"Send API enables dynamic parallel node dispatch",   "score":0.88},
        {"chunk_id":"C4","text":"Recipe for pasta carbonara requires eggs and cheese","score":0.08},
        {"chunk_id":"C5","text":"Checkpointers enable persistence across graph runs", "score":0.76},
        {"chunk_id":"C6","text":"Stock price of AAPL closed at $189 yesterday",      "score":0.31},
    ]
    print(f"📥 Retrieved {len(chunks)} chunks for query: '{state['query']}'")
    return {"chunks": chunks}

# ── Dispatch: one worker per chunk ────────────────────────
def dispatch_scorers(state: RAGState) -> list[Send]:
    print(f"\n⚡ Dispatching {len(state['chunks'])} chunk scorers in parallel...")
    return [
        Send("score_chunk", {
            "query":    state["query"],
            "chunk_id": c["chunk_id"],
            "text":     c["text"],
            "score":    c["score"],
        })
        for c in state["chunks"]
    ]

# ── Worker: scores chunk AND routes to different next nodes ─
def score_chunk(state: ChunkState) -> Command[Literal["use_chunk", "discard_chunk"]]:
    """
    Worker receives one chunk via Send().
    Based on score, routes to a DIFFERENT downstream node via Command.
    → score >= 0.6  : use_chunk   (high quality)
    → score <  0.6  : discard_chunk (low quality)
    This is the KEY insight: Send + Command together = dynamic fan-out
    with per-item routing to different nodes.
    """
    threshold = 0.6
    if state["score"] >= threshold:
        print(f"  ✅ {state['chunk_id']} score={state['score']:.2f} → use_chunk")
        return Command(
            update={"used": [f"{state['chunk_id']}: {state['text'][:45]}..."]},
            goto="use_chunk",
        )
    else:
        print(f"  ❌ {state['chunk_id']} score={state['score']:.2f} → discard_chunk")
        return Command(
            update={"discarded": [f"{state['chunk_id']}: score too low ({state['score']:.2f})"]},
            goto="discard_chunk",
        )

def use_chunk(state: RAGState):
    print(f"  📎 use_chunk: adding to context window")
    return {}

def discard_chunk(state: RAGState):
    print(f"  🗑  discard_chunk: dropping from context")
    return {}

def generate_answer(state: RAGState):
    print(f"\n🤖 Generating answer from {len(state['used'])} high-quality chunks")
    print(f"   Used    : {[c.split(':')[0] for c in state['used']]}")
    print(f"   Discarded: {[c.split(':')[0] for c in state['discarded']]}")
    answer = f"Answer using {len(state['used'])} relevant chunks: [LLM response here]"
    return {"answer": answer}

builder = StateGraph(RAGState)
builder.add_node("retrieve_chunks", retrieve_chunks)
builder.add_node("score_chunk",     score_chunk)
builder.add_node("use_chunk",       use_chunk)
builder.add_node("discard_chunk",   discard_chunk)
builder.add_node("generate_answer", generate_answer)
builder.add_edge(START,              "retrieve_chunks")
builder.add_conditional_edges("retrieve_chunks", dispatch_scorers, ["score_chunk"])
# score_chunk uses Command(goto=...) internally, so no edge needed from it
builder.add_edge("use_chunk",       "generate_answer")
builder.add_edge("discard_chunk",   "generate_answer")
builder.add_edge("generate_answer", END)
graph = builder.compile()

graph.invoke({
    "query":     "How does LangGraph handle state persistence?",
    "chunks":    [],
    "used":      [],
    "discarded": [],
    "answer":    "",
})

📥 Retrieved 6 chunks for query: 'How does LangGraph handle state persistence?'

⚡ Dispatching 6 chunk scorers in parallel...
  ✅ C1 score=0.91 → use_chunk
  ❌ C2 score=0.12 → discard_chunk
  ✅ C3 score=0.88 → use_chunk
  ❌ C4 score=0.08 → discard_chunk
  ✅ C5 score=0.76 → use_chunk
  ❌ C6 score=0.31 → discard_chunk
  🗑  discard_chunk: dropping from context
  📎 use_chunk: adding to context window

🤖 Generating answer from 3 high-quality chunks
   Used    : ['C1', 'C3', 'C5']
   Discarded: ['C2', 'C4', 'C6']


{'query': 'How does LangGraph handle state persistence?',
 'chunks': [{'chunk_id': 'C1',
   'text': 'LangGraph supports stateful multi-agent workflows',
   'score': 0.91},
  {'chunk_id': 'C2',
   'text': 'The weather in Paris is 22 degrees today',
   'score': 0.12},
  {'chunk_id': 'C3',
   'text': 'Send API enables dynamic parallel node dispatch',
   'score': 0.88},
  {'chunk_id': 'C4',
   'text': 'Recipe for pasta carbonara requires eggs and cheese',
   'score': 0.08},
  {'chunk_id': 'C5',
   'text': 'Checkpointers enable persistence across graph runs',
   'score': 0.76},
  {'chunk_id': 'C6',
   'text': 'Stock price of AAPL closed at $189 yesterday',
   'score': 0.31}],
 'used': ['C1: LangGraph supports stateful multi-agent workf...',
  'C3: Send API enables dynamic parallel node dispat...',
  'C5: Checkpointers enable persistence across graph...'],
 'discarded': ['C2: score too low (0.12)',
  'C4: score too low (0.08)',
  'C6: score too low (0.31)'],
 'answer': 'Answer using 3 releva

## 3. Subgraphs for Multi-Agent Systems

Subgraphs allow you to create modular, reusable workflows.

### Pattern:
```
Parent Graph:
  START → coordinator → [Subgraph 1, Subgraph 2] → aggregator → END

Subgraph 1:
  entry → process → exit

Subgraph 2:
  entry → analyze → exit
```

In [10]:
# Define a reusable subgraph for an agent
class AgentState(TypedDict):
    task: str
    result: str

def researcher_node(state: AgentState) -> dict:
    """Research agent subgraph."""
    task = state["task"]
    print(f"🔍 Researcher working on: {task}")
    return {"result": f"Research findings for: {task}"}

def analyzer_node(state: AgentState) -> dict:
    """Analyzer agent subgraph."""
    task = state["task"]
    print(f"📊 Analyzer working on: {task}")
    return {"result": f"Analysis of: {task}"}

# Build subgraphs
researcher_graph = StateGraph(AgentState)
researcher_graph.add_node("research", researcher_node)
researcher_graph.add_edge(START, "research")
researcher_graph.add_edge("research", END)
researcher_subgraph = researcher_graph.compile()

analyzer_graph = StateGraph(AgentState)
analyzer_graph.add_node("analyze", analyzer_node)
analyzer_graph.add_edge(START, "analyze")
analyzer_graph.add_edge("analyze", END)
analyzer_subgraph = analyzer_graph.compile()

print("✅ Subgraphs created!")

✅ Subgraphs created!


In [17]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# ── Subgraph A: Preprocessing Pipeline ────────────────────
# Has its own isolated state with private keys
class PreprocState(TypedDict):
    text:      str   # shared with parent
    char_count:int   # private — only inside subgraph A

def clean_text(state: PreprocState):
    cleaned = state["text"].strip().lower()
    print(f"  [SubA] clean_text: '{cleaned[:30]}...'")
    return {"text": cleaned}

def count_chars(state: PreprocState):
    n = len(state["text"])
    print(f"  [SubA] count_chars: {n} chars")
    return {"char_count": n}

preproc_builder = StateGraph(PreprocState)
preproc_builder.add_node("clean_text",  clean_text)
preproc_builder.add_node("count_chars", count_chars)
preproc_builder.add_edge(START,          "clean_text")
preproc_builder.add_edge("clean_text",  "count_chars")
preproc_builder.add_edge("count_chars", END)
preproc_subgraph = preproc_builder.compile()   # ← compiled = ready to be a node

# ── Subgraph B: Analysis Pipeline ─────────────────────────
class AnalysisState(TypedDict):
    text:      str    # shared with parent
    keywords: list[str]  # private — only inside subgraph B

def extract_keywords(state: AnalysisState):
    words    = [w for w in state["text"].split() if len(w) > 5]
    print(f"  [SubB] extract_keywords: {words[:4]}")
    return {"keywords": words}

def score_sentiment(state: AnalysisState):
    pos_words = ["great", "good", "excellent", "amazing", "fantastic"]
    score     = sum(1 for w in state["keywords"] if w in pos_words)
    label     = "positive" if score > 0 else "neutral"
    # Only 'text' is shared back to parent — 'keywords' stays private
    print(f"  [SubB] score_sentiment: {label}")
    return {"text": f"[{label.upper()}] {state['text']}"}

analysis_builder = StateGraph(AnalysisState)
analysis_builder.add_node("extract_keywords", extract_keywords)
analysis_builder.add_node("score_sentiment",  score_sentiment)
analysis_builder.add_edge(START,               "extract_keywords")
analysis_builder.add_edge("extract_keywords", "score_sentiment")
analysis_builder.add_edge("score_sentiment",  END)
analysis_subgraph = analysis_builder.compile()

# ── Parent Graph ───────────────────────────────────────────
# Uses both subgraphs as plain nodes via add_node()
class ParentState(TypedDict):
    text:   str   # shared key — subgraphs read & write this
    result: str

def format_result(state: ParentState):
    print(f"  [Parent] Formatting result.")
    return {"result": f"DONE: {state['text']}"}

parent = StateGraph(ParentState)
parent.add_node("preprocess", preproc_subgraph)    # ← subgraph as node!
parent.add_node("analyse",     analysis_subgraph)   # ← subgraph as node!
parent.add_node("format",      format_result)
parent.add_edge(START,         "preprocess")
parent.add_edge("preprocess",  "analyse")
parent.add_edge("analyse",     "format")
parent.add_edge("format",      END)
graph = parent.compile()

print("🚀 Running parent graph with 2 subgraphs...\n")
out = graph.invoke({"text": "  LangGraph is Great for building Agentic AI systems!  ", "result": ""})
print(f"\n✅ {out['result']}")

🚀 Running parent graph with 2 subgraphs...

  [SubA] clean_text: 'langgraph is great for buildin...'
  [SubA] count_chars: 51 chars
  [SubB] extract_keywords: ['langgraph', 'building', 'agentic', 'systems!']
  [SubB] score_sentiment: neutral
  [Parent] Formatting result.

✅ DONE: [NEUTRAL] langgraph is great for building agentic ai systems!


## 4. Advanced Project: Multi-Agent Research System

Build a system with multiple specialized agents working together:

- **Researcher**: Gathers information
- **Analyzer**: Evaluates credibility
- **Synthesizer**: Combines findings
- **Editor**: Formats output

In [11]:
from langgraph.graph.message import add_messages

class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    research_results: list
    analysis_results: list
    final_report: str

# Create specialized LLMs for each agent
researcher_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.3)
analyzer_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
synthesizer_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.5)

# coordinator becomes a pure edge function (no node registration)
def coordinator(state: MultiAgentState):
    query = state["messages"][-1].content
    print(f"🎯 Coordinator: Distributing task: {query}")
    return [
        Send("researcher", state),
        Send("analyzer", state)
    ]

def researcher(state: MultiAgentState) -> dict:
    query = state["messages"][-1].content
    print("🔬 Researcher: Gathering information...")
    response = researcher_llm.invoke([
        HumanMessage(content=f"Research this topic and provide key facts: {query}")
    ])
    return {"research_results": [response.content]}

def analyzer(state: MultiAgentState) -> dict:
    query = state["messages"][-1].content
    print("📊 Analyzer: Evaluating sources...")
    response = analyzer_llm.invoke([
        HumanMessage(content=f"Analyze the credibility and relevance of information about: {query}")
    ])
    return {"analysis_results": [response.content]}

def synthesizer(state: MultiAgentState) -> dict:
    """Synthesize all findings into a coherent report."""
    print("✍️  Synthesizer: Creating final report...")
    
    research = "\n".join(state.get("research_results", []))
    analysis = "\n".join(state.get("analysis_results", []))
    
    response = synthesizer_llm.invoke([
        HumanMessage(content=f"Synthesize these findings into a coherent report:\n\nResearch:\n{research}\n\nAnalysis:\n{analysis}")
    ])
    
    return {"final_report": response.content}

# Build multi-agent graph
multi_agent_builder = StateGraph(MultiAgentState)
# Remove: multi_agent_builder.add_node("coordinator", coordinator)
multi_agent_builder.add_node("researcher", researcher)
multi_agent_builder.add_node("analyzer", analyzer)
multi_agent_builder.add_node("synthesizer", synthesizer)

# coordinator goes here as the edge function from START
multi_agent_builder.add_conditional_edges(START, coordinator, ["researcher", "analyzer"])
multi_agent_builder.add_edge("researcher", "synthesizer")
multi_agent_builder.add_edge("analyzer", "synthesizer")
multi_agent_builder.add_edge("synthesizer", END)

multi_agent_system = multi_agent_builder.compile()

print("✅ Multi-agent research system ready!")

✅ Multi-agent research system ready!


In [18]:
# Test the multi-agent system
result = multi_agent_system.invoke({
    "messages": [HumanMessage(content="Explain the impact of quantum computing on cryptography")],
    "research_results": [],
    "analysis_results": [],
    "final_report": ""
})

print("\n" + "=" * 70)
print("📄 Final Research Report")
print("=" * 70)
print(result["final_report"])
print("=" * 70)

🎯 Coordinator: Distributing task: Explain the impact of quantum computing on cryptography
🔬 Researcher: Gathering information...
📊 Analyzer: Evaluating sources...
✍️  Synthesizer: Creating final report...

📄 Final Research Report
**Quantum Computing and Its Impact on Cryptography**

### Executive Summary
Quantum computing is poised to transform the field of cryptography, offering both unprecedented challenges and opportunities. This technology leverages quantum mechanics to perform computations at speeds unattainable by classical computers, presenting a significant threat to traditional cryptographic systems but also paving the way for more secure communication protocols like Quantum Key Distribution (QKD).

### Introduction
The evolution of quantum computing has critical implications for cryptography, the science of securing communication through coding. This report synthesizes current research and analysis on the impact of quantum computing on cryptographic practices, assessing both 

In [19]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

# ── Shared state across all agents ────────────────────────
class AgentState(TypedDict):
    task:     str
    log:      Annotated[list[str], operator.add]
    result:   str

# ── Specialist Sub-Agents (each is a subgraph) ────────────

# Sub-Agent 1: Web Research Agent
class WebState(TypedDict):
    task:   str
    log:    Annotated[list[str], operator.add]
    result: str

def web_search(state: WebState):
    print("  🌐 WebAgent: searching the web...")
    return {"log": ["web_search completed"]}

def web_summarize(state: WebState):
    print("  🌐 WebAgent: summarizing results...")
    summary = f"[WEB RESULT] Found 5 sources about: {state['task']}"
    return {"result": summary, "log": ["web_summarize completed"]}

web_builder = StateGraph(WebState)
web_builder.add_node("web_search",    web_search)
web_builder.add_node("web_summarize", web_summarize)
web_builder.add_edge(START,            "web_search")
web_builder.add_edge("web_search",    "web_summarize")
web_builder.add_edge("web_summarize", END)
web_agent = web_builder.compile()

# Sub-Agent 2: Code Agent
class CodeState(TypedDict):
    task:   str
    log:    Annotated[list[str], operator.add]
    result: str

def write_code(state: CodeState):
    print("  💻 CodeAgent: writing code...")
    return {"log": ["write_code completed"]}

def test_code(state: CodeState):
    print("  💻 CodeAgent: running tests...")
    code_out = f"[CODE RESULT] Python solution for: {state['task']}"
    return {"result": code_out, "log": ["test_code completed"]}

code_builder = StateGraph(CodeState)
code_builder.add_node("write_code", write_code)
code_builder.add_node("test_code",  test_code)
code_builder.add_edge(START,        "write_code")
code_builder.add_edge("write_code", "test_code")
code_builder.add_edge("test_code",  END)
code_agent = code_builder.compile()

# ── Orchestrator Node ─────────────────────────────────────
# Reads the task, decides which specialist to route to
def orchestrator(state: AgentState) -> Command[Literal["web_agent", "code_agent"]]:
    task = state["task"].lower()
    print(f"🎯 Orchestrator received: '{state['task']}'")

    # Simple rule-based routing (would be an LLM in production)
    if "code" in task or "python" in task or "function" in task:
        print("  → Routing to CODE AGENT")
        return Command(update={"log": ["orchestrator: delegated to code_agent"]},
                       goto="code_agent")
    else:
        print("  → Routing to WEB AGENT")
        return Command(update={"log": ["orchestrator: delegated to web_agent"]},
                       goto="web_agent")

def synthesizer(state: AgentState):
    print(f"\n📋 Synthesizer received result: {state['result'][:50]}...")
    print(f"   Full log: {state['log']}")
    return {}

# ── Parent / Orchestrator Graph ────────────────────────────
builder = StateGraph(AgentState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("web_agent",     web_agent)   # ← subgraph!
builder.add_node("code_agent",    code_agent)  # ← subgraph!
builder.add_node("synthesizer",   synthesizer)
builder.add_edge(START,           "orchestrator")
builder.add_edge("web_agent",     "synthesizer")
builder.add_edge("code_agent",    "synthesizer")
builder.add_edge("synthesizer",   END)
graph = builder.compile()

# ── Test both routing paths ───────────────────────────────
for task in ["Research latest AI trends in 2025",
              "Write a Python function to sort a list"]:
    print(f"\n{'='*55}\n📥 Task: {task}")
    graph.invoke({"task": task, "log": [], "result": ""})


📥 Task: Research latest AI trends in 2025
🎯 Orchestrator received: 'Research latest AI trends in 2025'
  → Routing to WEB AGENT
  🌐 WebAgent: searching the web...
  🌐 WebAgent: summarizing results...

📋 Synthesizer received result: [WEB RESULT] Found 5 sources about: Research lates...
   Full log: ['orchestrator: delegated to web_agent', 'orchestrator: delegated to web_agent', 'web_search completed', 'web_summarize completed']

📥 Task: Write a Python function to sort a list
🎯 Orchestrator received: 'Write a Python function to sort a list'
  → Routing to CODE AGENT
  💻 CodeAgent: writing code...
  💻 CodeAgent: running tests...

📋 Synthesizer received result: [CODE RESULT] Python solution for: Write a Python ...
   Full log: ['orchestrator: delegated to code_agent', 'orchestrator: delegated to code_agent', 'write_code completed', 'test_code completed']


## Map Reduce Patterns

```mermaid
flowchart TD

    %% ─── PHASE 0  ·  Initial State ──────────────────────────────
    OS1["🌐 OverallState  ·  initial
    ─────────────────────────────────
    topic: 'Animals of the world'
    subjects:  []
    jokes:     Annotated[list, operator.add]
    best_joke: ''"]

    %% ─── PHASE 1  ·  Regular node populates subjects ────────────
    GT["generate_topics
    ─────────────────────────────────
    Plain node — returns a dict
    subjects: ['lions','elephants','penguins']"]

    OS1 -->|"graph.invoke(...)"| GT

    %% ─── PHASE 2  ·  Conditional edge dispatches workers ─────────
    DISP["dispatch_to_jokes
    ─────────────────────────────────
    Conditional edge function
    returns  [ Send(...), Send(...), Send(...) ]"]

    GT -->|"OverallState.subjects populated"| DISP

    %% ─── MAP PHASE  ·  3 workers fire in parallel ────────────────
    subgraph MAP["⚡  MAP PHASE  ·  all 3 workers run simultaneously"]
        direction LR

        WS1["WorkerState 1
        subject: 'lions'"]
        WS2["WorkerState 2
        subject: 'elephants'"]
        WS3["WorkerState 3
        subject: 'penguins'"]

        GJ1["generate_joke
        worker 1"]
        GJ2["generate_joke
        worker 2"]
        GJ3["generate_joke
        worker 3"]

        WS1 --> GJ1
        WS2 --> GJ2
        WS3 --> GJ3
    end

    DISP -->|"Send · subject: 'lions'"| WS1
    DISP -->|"Send · subject: 'elephants'"| WS2
    DISP -->|"Send · subject: 'penguins'"| WS3

    %% ─── REDUCE PHASE  ·  fan-in merges all results ─────────────
    RD["🔀 REDUCE PHASE  ·  Annotated[list, operator.add]
    ─────────────────────────────────
    operator.add appends all 3 worker results
    jokes = joke_1 + joke_2 + joke_3"]

    GJ1 -->|"return  jokes: ['Why lions...']"| RD
    GJ2 -->|"return  jokes: ['Elephants fear...']"| RD
    GJ3 -->|"return  jokes: ['Penguins ice...']"| RD

    %% ─── PHASE 4  ·  Final node reads merged list ────────────────
    BJ["best_joke  ·  final node
    ─────────────────────────────────
    Reads fully merged jokes list
    Picks the best one"]

    RD -->|"all workers done — OverallState.jokes has 3 items"| BJ

    OS2["🌐 OverallState  ·  final
    ─────────────────────────────────
    jokes:           3 items collected
    best_selected_joke: 'penguins'"]

    BJ -->|"return  best_selected_joke: 'penguins'"| OS2
    OS2 --> STOP(["END"])

    %% ─── KEY RULE ANNOTATIONS ────────────────────────────────────
    KEY1["KEY  Send API
    ─────────────────
    Each Send injects its own
    private WorkerState —
    workers never share state"]

    KEY2["KEY  operator.add reducer
    ─────────────────
    results: list append-only
    safe for parallel writes
    no race conditions"]

    DISP -.->|"teaches"| KEY1
    RD   -.->|"teaches"| KEY2

    %% ─── STYLES ──────────────────────────────────────────────────
    style OS1   fill:#EEEDFE,stroke:#534AB7,color:#26215C
    style OS2   fill:#EEEDFE,stroke:#534AB7,color:#26215C
    style STOP  fill:#EEEDFE,stroke:#534AB7,color:#26215C
    style GT    fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style BJ    fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style DISP  fill:#F1EFE8,stroke:#5F5E5A,color:#2C2C2A
    style WS1   fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style WS2   fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style WS3   fill:#E1F5EE,stroke:#0F6E56,color:#04342C
    style GJ1   fill:#FAECE7,stroke:#993C1D,color:#4A1B0C
    style GJ2   fill:#FAECE7,stroke:#993C1D,color:#4A1B0C
    style GJ3   fill:#FAECE7,stroke:#993C1D,color:#4A1B0C
    style RD    fill:#FAEEDA,stroke:#854F0B,color:#412402
    style KEY1  fill:#F1EFE8,stroke:#888780,color:#444441
    style KEY2  fill:#F1EFE8,stroke:#888780,color:#444441
    style MAP   fill:#FFF8F6,stroke:#F0997B,color:#993C1D
```

### Colour legend

| Colour | Meaning |
|--------|---------|
| Purple | `OverallState` — global shared state |
| Teal | Regular nodes and `WorkerState` |
| Coral | Worker nodes (MAP workers) |
| Amber | Reducer — `Annotated[list, operator.add]` |
| Gray | Dispatch / conditional edge functions |
| Dashed arrow | Conceptual annotation / key insight |

### The two rules that make Map-Reduce work

1. **`Send("node", {custom_state})`** fans out — each call spawns one parallel worker with its own private state.
2. **`Annotated[list, operator.add]`** fans in — the reducer safely appends every worker's result list into one, with no race conditions.

In [5]:
# ============================================================
# MAP-REDUCE IN LANGGRAPH
# MAP phase  : Send API fans out N parallel workers (one per item).
# REDUCE phase: Annotated[list, operator.add] reducer collects all results.
# The reduce node only runs AFTER all N map workers complete.
# Classic pattern: generate → score → pick best.
# Example: Generate 4 essay outlines in parallel, pick the best one.
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# ── State Design ─────────────────────────────────────────
class OverallState(TypedDict):
    topic:       str
    approaches: list[str]       # the N items to fan-out over
    # Annotated + operator.add is the REDUCE mechanism
    # All worker outputs append to this list automatically
    outlines:   Annotated[list[dict], operator.add]
    best_outline:str

class WorkerState(TypedDict):
    topic:    str
    approach: str    # each worker gets a different approach

# ── PHASE 1: Generate N approaches (the setup) ────────────
def generate_approaches(state: OverallState):
    """Generates the list of items to map over."""
    approaches = [
        "chronological narrative",
        "problem-solution framework",
        "compare and contrast",
        "case study based",
    ]
    print(f"📋 Generated {len(approaches)} essay approaches to evaluate in parallel.")
    return {"approaches": approaches}

# ── MAP: Dispatch one worker per approach via Send() ──────
def dispatch_workers(state: OverallState) -> list[Send]:
    """MAP: Fan-out — one Send per approach = N parallel workers."""
    return [
        Send("write_outline", {"topic": state["topic"], "approach": a})
        for a in state["approaches"]
    ]

# ── Worker node: runs N times in parallel ─────────────────
def write_outline(state: WorkerState):
    """Each worker creates one outline with a score."""
    approach = state["approach"]
    topic    = state["topic"]

    # Simulate scoring (would be an LLM evaluator in production)
    scores = {
        "chronological narrative":    72,
        "problem-solution framework":  91,
        "compare and contrast":        68,
        "case study based":            85,
    }
    score = scores.get(approach, 50)
    outline = {
        "approach": approach,
        "score":    score,
        "preview":  f"Essay on '{topic}' using {approach}",
    }
    print(f"  ✍️  Worker [{approach}] → score: {score}")
    return {"outlines": [outline]}  # ← appended to OverallState.outlines

# ── REDUCE: Runs after ALL workers finish ─────────────────
def pick_best_outline(state: OverallState):
    """
    REDUCE phase: receives ALL outlines merged by operator.add.
    Picks the highest-scoring one.
    """
    print(f"\n🔀 REDUCE: received {len(state['outlines'])} outlines.")
    best = max(state["outlines"], key=lambda x: x["score"])
    print(f"🏆 Best approach: '{best['approach']}' (score: {best['score']})")
    return {"best_outline": best["preview"]}

# ── Build Graph ───────────────────────────────────────────
builder = StateGraph(OverallState)
builder.add_node("generate_approaches", generate_approaches)
builder.add_node("write_outline",       write_outline)       # MAP worker
builder.add_node("pick_best_outline",   pick_best_outline)   # REDUCE
builder.add_edge(START,                   "generate_approaches")

# Conditional edge returns [Send(...),...] = the MAP phase
builder.add_conditional_edges("generate_approaches", dispatch_workers, ["write_outline"])

# All workers → reduce node (fan-in)
builder.add_edge("write_outline",       "pick_best_outline")
builder.add_edge("pick_best_outline",   END)
graph = builder.compile()

print("🚀 Running map-reduce essay outliner...\n")
result = graph.invoke({
    "topic":        "The Impact of Agentic AI on Software Engineering",
    "approaches":  [],
    "outlines":    [],
    "best_outline":"",
})
print(f"\n✅ Best: {result['best_outline']}")

🚀 Running map-reduce essay outliner...

📋 Generated 4 essay approaches to evaluate in parallel.
  ✍️  Worker [chronological narrative] → score: 72
  ✍️  Worker [problem-solution framework] → score: 91
  ✍️  Worker [compare and contrast] → score: 68
  ✍️  Worker [case study based] → score: 85

🔀 REDUCE: received 4 outlines.
🏆 Best approach: 'problem-solution framework' (score: 91)

✅ Best: Essay on 'The Impact of Agentic AI on Software Engineering' using problem-solution framework


## 🆕 Deferred Nodes — Race-Condition-Free Aggregation

In map-reduce patterns, there's a race condition: the aggregator node might start before all parallel branches finish. **Deferred nodes** solve this by waiting for ALL upstream parallel paths to complete before executing.

```python
graph.add_node("synthesizer", synthesize_fn, defer=True)
```

Without `defer=True`, the synthesizer might run before all researchers finish. With it, LangGraph guarantees it only runs after ALL parallel Send() executions complete.

## Deferred Nodes in LangGraph

### Diagram 1 — Graph structure: two branches of unequal length

```mermaid
flowchart TD

    %% ─── Entry ───────────────────────────────────────────────────
    START(["__start__"])
    A["node_a
    fans out to both branches simultaneously"]

    %% ─── Branch 1: Short path (2 hops to D) ─────────────────────
    B["node_b
    Branch 1  ·  arrives at D in 2 hops"]

    %% ─── Branch 2: Long path (3 hops to D) ──────────────────────
    C["node_c
    Branch 2  ·  step 1 of 2"]

    C2["node_c2
    Branch 2  ·  step 2 of 2  ·  arrives at D in 3 hops"]

    %% ─── Deferred node ───────────────────────────────────────────
    D["node_d
    defer=True
    waits for ALL incoming branches before running"]

    END_(["__end__"])

    START --> A
    A -->|"fans out"| B
    A -->|"fans out"| C
    C --> C2
    B -.->|"arrives at super-step 3"| D
    C2 -.->|"arrives at super-step 4"| D
    D --> END_

    style START fill:#F1EFE8,stroke:#5F5E5A,color:#2C2C2A
    style END_  fill:#F1EFE8,stroke:#5F5E5A,color:#2C2C2A
    style A     fill:#F1EFE8,stroke:#5F5E5A,color:#2C2C2A
    style B     fill:#FAECE7,stroke:#993C1D,color:#712B13
    style C     fill:#E1F5EE,stroke:#0F6E56,color:#085041
    style C2    fill:#E1F5EE,stroke:#0F6E56,color:#085041
    style D     fill:#EEEDFE,stroke:#534AB7,color:#26215C
```

> Dashed arrows show the **problem**: B→D and C2→D arrive at `node_d`
> in **different super-steps** because the branches have unequal lengths.

---

### Diagram 2 — Execution comparison: without vs with `defer=True`

```mermaid
flowchart LR

    subgraph NO["Without  defer=True  —  node_d runs TWICE"]
        direction TB
        n1["Step 1  ·  node_a runs"]
        n2["Step 2  ·  node_b and node_c run in parallel"]
        n3["Step 3  ·  node_b finishes
        node_c2 starts running
        node_d fires immediately  — INCOMPLETE STATE"]
        n4["Step 4  ·  node_c2 finishes
        node_d fires AGAIN  — double execution bug"]
        n1 --> n2 --> n3 --> n4
    end

    subgraph YES["With  defer=True  —  node_d runs ONCE"]
        direction TB
        y1["Step 1  ·  node_a runs"]
        y2["Step 2  ·  node_b and node_c run in parallel"]
        y3["Step 3  ·  node_b finishes
        node_c2 starts running
        node_d waits  — defer holds it"]
        y4["Step 4  ·  node_c2 finishes
        node_d runs ONCE with complete state"]
        y1 --> y2 --> y3 --> y4
    end

    style n3 fill:#FCEBEB,stroke:#A32D2D,color:#501313
    style n4 fill:#FCEBEB,stroke:#A32D2D,color:#501313
    style y3 fill:#FAEEDA,stroke:#854F0B,color:#412402
    style y4 fill:#EAF3DE,stroke:#3B6D11,color:#173404
    style NO  fill:#FFF5F5,stroke:#F09595,color:#A32D2D
    style YES fill:#F2FCF6,stroke:#5DCAA5,color:#0F6E56
```

---

### Diagram 3 — The entire fix is one keyword

```mermaid
flowchart LR

    WRONG["builder.add_node
    ──────────────────────────
    add_node('node_d', node_d)
    ──────────────────────────
    node_d executes TWICE
    incomplete state each time"]

    RIGHT["builder.add_node  with defer
    ──────────────────────────
    add_node('node_d', node_d,
             defer=True)
    ──────────────────────────
    node_d executes ONCE
    complete state  — correct"]

    WRONG -->|"add  defer=True"| RIGHT

    style WRONG fill:#FCEBEB,stroke:#A32D2D,color:#501313
    style RIGHT fill:#EAF3DE,stroke:#3B6D11,color:#173404
```

---

### When to use `defer=True`

| Situation | Use `defer=True`? | Why |
|-----------|:-----------------:|-----|
| Equal-length parallel branches | No | Normal fan-in handles it |
| **Unequal-length branches to same node** | **Yes** | Without it, node fires per branch arrival |
| Single incoming edge | Never | Only one path — no conflict possible |
| Map-Reduce (Send API workers) | No | All workers are the same length |

> **The rule in one sentence:**
> Add `defer=True` whenever a node receives edges from parallel branches
> that are **not the same number of hops** from their common ancestor.

In [4]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class ResearchState(TypedDict):
    topics: list[str]
    findings: Annotated[list[str], operator.add]  # accumulated from parallel branches
    final_report: str

def coordinator(state: ResearchState) -> list[Send]:
    """Distribute research topics in parallel."""
    print(f"📋 Coordinating research on {len(state['topics'])} topics")
    return [Send("research_topic", {"topics": [topic], "findings": [], "final_report": ""})
            for topic in state["topics"]]

def research_topic(state: ResearchState) -> dict:
    """Research a single topic (runs in parallel)."""
    topic = state["topics"][0]
    finding = f"Research finding for '{topic}': key insight discovered"
    print(f"🔬 Researching: {topic}")
    return {"findings": [finding]}

def synthesize_report(state: ResearchState) -> dict:
    """Synthesize all findings — runs AFTER all parallel branches complete."""
    print(f"\n📊 Synthesizing {len(state['findings'])} findings...")
    findings_text = "\n".join(f"• {f}" for f in state["findings"])
    report = "FINAL REPORT:\n" + findings_text
    return {"final_report": report}

# Build graph
builder = StateGraph(ResearchState)
# Remove: builder.add_node("coordinator", coordinator)
builder.add_node("research_topic", research_topic)
builder.add_node("synthesizer", synthesize_report, defer=True)

# coordinator is the conditional edge function from START, not a node
builder.add_conditional_edges(START, coordinator, ["research_topic"])
builder.add_edge("research_topic", "synthesizer")
builder.add_edge("synthesizer", END)

app = builder.compile()

result = app.invoke({
    "topics": ["AI Safety", "Multi-Agent Systems", "LangGraph Architecture"],
    "findings": [],
    "final_report": ""
})
print(result["final_report"])

📋 Coordinating research on 3 topics
🔬 Researching: AI Safety
🔬 Researching: Multi-Agent Systems
🔬 Researching: LangGraph Architecture

📊 Synthesizing 3 findings...
FINAL REPORT:
• Research finding for 'AI Safety': key insight discovered
• Research finding for 'Multi-Agent Systems': key insight discovered
• Research finding for 'LangGraph Architecture': key insight discovered


**Why `defer=True` matters**: Without it, if topic A finishes first, `synthesizer` might run with only 1 finding. With `defer=True`, LangGraph queues the synthesizer and only executes it when ALL parallel `research_topic` branches complete — guaranteed.

In [6]:
# ============================================================
# DEFERRED NODES  (defer=True)
# Problem: In complex branching graphs, a node may have MULTIPLE
# incoming paths of different lengths.
# Without defer: node runs as soon as the FIRST branch reaches it.
# With defer  : node waits until ALL branches have completed first.
#
# The key difference from regular fan-in:
# Regular fan-in (same-length branches) → works without defer.
# Unequal-length branches → you NEED defer=True.
#
# Example: A → B (fast, 1 step) and A → C → D (slow, 2 steps)
# Both paths end at node E. Without defer, E runs when B finishes
# before D even starts. With defer=True, E waits for both.
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list[str], operator.add]

# ── The graph has UNEQUAL-LENGTH parallel branches ─────────
#
#  START → A ──→ B ──────────────────────→ D (deferred)
#              └→ C → C2 ────────────────→ D (deferred)
#
# Branch 1: A → B → D         (2 hops to D)
# Branch 2: A → C → C2 → D   (3 hops to D)
# D must wait for BOTH branches before running.

def node_a(state: State):
    print(f"  [A] running | state={state['aggregate']}")
    return {"aggregate": ["A"]}

def node_b(state: State):
    # Short branch — arrives at D after 1 step
    print(f"  [B] running | state={state['aggregate']}")
    return {"aggregate": ["B"]}

def node_c(state: State):
    # Long branch step 1 — has more work to do
    print(f"  [C] running | state={state['aggregate']}")
    return {"aggregate": ["C"]}

def node_c2(state: State):
    # Long branch step 2 — arrives at D after 2 steps
    print(f"  [C2] running | state={state['aggregate']}")
    return {"aggregate": ["C2"]}

def node_d(state: State):
    """
    This node is DEFERRED — waits for ALL incoming branches.
    It only runs once B AND C2 have both completed.
    state['aggregate'] will contain A, B, C, C2 when this runs.
    """
    print(f"  [D] DEFERRED node running. Got all: {state['aggregate']}")
    return {"aggregate": ["D"]}

# ── Without defer (WRONG behavior) ───────────────────────
print("=== WITHOUT defer=True (D runs too early!) ===")
b1 = StateGraph(State)
b1.add_node("a",  node_a)
b1.add_node("b",  node_b)
b1.add_node("c",  node_c)
b1.add_node("c2", node_c2)
b1.add_node("d",  node_d)            # ← NO defer, runs early
b1.add_edge(START, "a")
b1.add_edge("a", "b"); b1.add_edge("a", "c")
b1.add_edge("b", "d"); b1.add_edge("c", "c2"); b1.add_edge("c2", "d")
b1.add_edge("d", END)
result_no_defer = b1.compile().invoke({"aggregate": []})
print(f"Result (no defer): {result_no_defer['aggregate']}")
print("  ↳ D ran twice! Once when B finished, once when C2 finished.\n")

# ── With defer=True (CORRECT behavior) ───────────────────
print("=== WITH defer=True (D waits for ALL branches) ===")
b2 = StateGraph(State)
b2.add_node("a",  node_a)
b2.add_node("b",  node_b)
b2.add_node("c",  node_c)
b2.add_node("c2", node_c2)
b2.add_node("d",  node_d, defer=True)  # ← defer=True is the ONLY change!
b2.add_edge(START, "a")
b2.add_edge("a", "b"); b2.add_edge("a", "c")
b2.add_edge("b", "d"); b2.add_edge("c", "c2"); b2.add_edge("c2", "d")
b2.add_edge("d", END)
result_deferred = b2.compile().invoke({"aggregate": []})
print(f"Result (deferred): {result_deferred['aggregate']}")
print("  ↳ D ran ONCE after A, B, C, C2 all completed. ✅")

=== WITHOUT defer=True (D runs too early!) ===
  [A] running | state=[]
  [B] running | state=['A']
  [C] running | state=['A']
  [C2] running | state=['A', 'B', 'C']
  [D] DEFERRED node running. Got all: ['A', 'B', 'C']
  [D] DEFERRED node running. Got all: ['A', 'B', 'C', 'C2', 'D']
Result (no defer): ['A', 'B', 'C', 'C2', 'D', 'D']
  ↳ D ran twice! Once when B finished, once when C2 finished.

=== WITH defer=True (D waits for ALL branches) ===
  [A] running | state=[]
  [B] running | state=['A']
  [C] running | state=['A']
  [C2] running | state=['A', 'B', 'C']
  [D] DEFERRED node running. Got all: ['A', 'B', 'C', 'C2']
Result (deferred): ['A', 'B', 'C', 'C2', 'D']
  ↳ D ran ONCE after A, B, C, C2 all completed. ✅


## 5. Key Takeaways

### Concepts Mastered

1. **Send API**: Parallel execution of nodes
2. **Subgraphs**: Modular, reusable workflows
3. **Multi-agent systems**: Specialized agents working together
4. **Map-reduce patterns**: Distribute, process, aggregate
5. **Deferred nodes**: Wait for all parallel tasks to complete

### Best Practices

✅ **Use Send for independent tasks** - Each can run in parallel  
✅ **Design subgraphs for reusability** - DRY principle  
✅ **Coordinate agents carefully** - Clear responsibilities  
✅ **Use reducers for aggregation** - `operator.add`, etc.  

### What's Next?

In **Notebook 07**, you'll learn production-ready patterns including caching, hooks, error handling, and deployment!